In [4]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score


In [5]:

# =========================
# 2. LOAD DATA
# =========================
fear_greed_path = r"C:\Users\Hanisha\Downloads\fear_greed_index.csv"
trades_path = r"C:\Users\Hanisha\Downloads\historical_data.csv"

sentiment = pd.read_csv(fear_greed_path)
trades = pd.read_csv(trades_path)


In [6]:

# =========================
# 3. CLEAN COLUMN NAMES
# =========================
trades.columns = trades.columns.str.strip()
sentiment.columns = sentiment.columns.str.strip()

# =========================
# 4. DATE CONVERSION (SAFE)
# =========================


In [7]:

trades['date'] = pd.to_datetime(
    trades['Timestamp IST'],
    dayfirst=True,
    errors='coerce'
).dt.date

sentiment['date'] = pd.to_datetime(
    sentiment['date'],
    dayfirst=True,
    errors='coerce'
).dt.date


In [8]:

# Drop invalid rows
trades = trades.dropna(subset=['date'])
sentiment = sentiment.dropna(subset=['date'])

# =========================
# 5. RENAME COLUMNS
# =========================
trades.rename(columns={
    'Account': 'account_id',
    'Closed PnL': 'pnl',
    'Size USD': 'trade_size',
    'Side': 'side'
}, inplace=True)

sentiment.rename(columns={
    'classification': 'sentiment'
}, inplace=True)


In [9]:

# =========================
# 6. FEATURE ENGINEERING
# =========================

# Win column
trades['win'] = (trades['pnl'] > 0).astype(int)

# Daily aggregation (VERY IMPORTANT)
daily = trades.groupby(['account_id', 'date']).agg({
    'pnl': 'sum',
    'trade_size': 'mean',
    'win': 'mean'
}).reset_index()

# Merge sentiment
daily = daily.merge(sentiment[['date', 'sentiment']], on='date', how='left')

# Drop missing sentiment rows
daily = daily.dropna(subset=['sentiment'])

# =========================


In [10]:
# 7. ENCODE FEATURES
# =========================

# Encode sentiment
daily['sentiment'] = daily['sentiment'].astype('category').cat.codes

# Target variable (profitability)
daily['target'] = (daily['pnl'] > 0).astype(int)

# Features
X = daily[['trade_size', 'win', 'sentiment']]
y = daily['target']

# =========================
# 8. TRAIN-TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [11]:

# =========================
# 9. MODEL TRAINING
# =========================
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)

# =========================
# 10. PREDICTION
# =========================
y_pred = model.predict(X_test)


In [12]:

# =========================
# 11. EVALUATION
# =========================
print("\n=== MODEL PERFORMANCE ===")

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))



=== MODEL PERFORMANCE ===
Accuracy: 0.9811320754716981

Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.95      0.98        43
           1       0.97      1.00      0.98        63

    accuracy                           0.98       106
   macro avg       0.98      0.98      0.98       106
weighted avg       0.98      0.98      0.98       106



In [13]:

# =========================
# 12. FEATURE IMPORTANCE
# =========================
importance = pd.Series(model.feature_importances_, index=X.columns)
print("\nFeature Importance:\n", importance.sort_values(ascending=False))

# =========================
# 13. SAMPLE PREDICTION
# =========================
sample = X_test.iloc[0:5]
print("\nSample Predictions:\n", model.predict(sample))


Feature Importance:
 win           0.892241
trade_size    0.086870
sentiment     0.020890
dtype: float64

Sample Predictions:
 [0 1 1 1 0]


# CLUSTERING TRADERS


In [14]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Features per trader
features = trades.groupby('account_id').agg({
    'pnl': 'mean',
    'trade_size': 'mean',
    'win': 'mean'
}).dropna()

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

# KMeans
kmeans = KMeans(n_clusters=3, random_state=42)
features['cluster'] = kmeans.fit_predict(X_scaled)

print(features.head())

                                                   pnl    trade_size  \
account_id                                                             
0x083384f897ee0f19899168e3b1bec365f52a9012  419.127768  16159.576734   
0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd    6.577654   1653.226327   
0x271b280974205ca63b716753467d5a371de622ab  -18.492043   8893.000898   
0x28736f43f1e871e6aa8b1148d38d4994275d72c4    9.951530    507.626933   
0x2c229d22b100a7beb69122eed721cee9b24011dd   52.071011   3138.894782   

                                                 win  cluster  
account_id                                                     
0x083384f897ee0f19899168e3b1bec365f52a9012  0.359612        2  
0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd  0.442720        1  
0x271b280974205ca63b716753467d5a371de622ab  0.301917        1  
0x28736f43f1e871e6aa8b1148d38d4994275d72c4  0.438585        1  
0x2c229d22b100a7beb69122eed721cee9b24011dd  0.519914        1  


# STREAMLIT DASHBOARD

In [16]:
!pip install streamlit

   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.1 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.1 MB 1.1 MB/s eta 0:00:08
   --- ------------------------------------ 0.8/9.1 MB 958.5 kB/s eta 0:00:09
   --- ------------------------------------ 0.8/9.1 MB 958.5 kB/s eta 0:00:09
   ---- ----------------------------------- 1.0/9.1 MB 931.8 kB/s eta 0:00:09
   ------ --------------------------------- 1.6/9.1 MB 1.1 MB/s eta 0:00:07
   --------- ------------------------------ 2.1/9.1 MB 1.3 MB/s eta 0:00:06
   ---------- ----------------------------- 2.4/9.1 MB 1.4 MB/s eta 0:00:05
   ------------- -------------------------- 3.1/9.1 MB 1.6 MB/s eta 0:00:04
   ------------------ --------------------- 4.2/9.1 MB 1.9 MB/s eta 0:00:03
   --------------------- ------------------ 5.0/9.1 MB 2.2 MB/s eta 0:00:02
   ------------------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
